## **Simple Rule Learners: ZeroR, OneR and PRISM**

Before complex rule learners, two **very simple** baselines show how much of a problem a rule can explain:

* **ZeroR** predicts the majority class ( the floor that any model must beat )
* **OneR** ( Holte, 1993 ) chooses the **single attribute** whose value → class table makes the fewest training errors
* **PRISM** ( Cendrowska, 1987 ) is a **covering algorithm**: for each class it builds rules one condition at a time, always adding the condition with the **highest precision**, and removes the covered cases

The three learners work on the **discretised** attributes ( *low / mid / high* ). This notebook implements them from scratch and evaluates them with the metrics of the series, a **cross-validation that refits the discretiser inside every fold**, bootstrap intervals and reference models.

**Notebooks of this series:** 1. CN2 Rule Induction · 2. Expert Systems · 3. Case-Based Reasoning · 4. Apriori for Classification ( CBA ) · 5. Simple Rule Learners ( ZeroR, OneR, PRISM ) · 6. RIPPER · 7. Rules from Decision Trees · 8. RuleFit · 9. Fuzzy Rule-Based Classifier

### **0. Environment Setup**

In [1]:
#Import Libs
import sklearn
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

In [2]:
#Define Colors Palette and Standards for Illustrations
RED, DARK, GRAY, LIGHT, SOFT = "#CD2027", "#404040", "#7F7F7F", "#D9D9D9", "#E9A0A3"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.2),
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titleweight": "bold",
    "axes.prop_cycle": cycler(color=[RED, DARK, SOFT, GRAY]),
})
pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)

In [3]:
#Helper Functions: Evaluation Reports ( Confusion Matrix, Metrics, ROC and Precision-Recall )
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, log_loss, matthews_corrcoef,
                             confusion_matrix, roc_curve, precision_recall_curve)

def plot_confusion(cm, labels, ax, title="Confusion matrix"):
    ax.imshow(cm, cmap="Reds"); ax.grid(False)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45 if len(labels) > 4 else 0, ha="right" if len(labels) > 4 else "center")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=10 if len(labels) > 4 else 13,
                    color="white" if cm[i, j] > cm.max() / 2 else DARK)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)

def report_binary(y_true, y_pred, y_score, name="model", labels=("negative", "positive"), plot=True):
    """Confusion matrix, ROC and Precision-Recall curves + a Series with the main metrics (positive class = 1)."""
    y_true, y_pred, y_score = np.asarray(y_true), np.asarray(y_pred), np.asarray(y_score)
    metrics = {"accuracy": accuracy_score(y_true, y_pred), "balanced acc.": balanced_accuracy_score(y_true, y_pred),
               "precision": precision_score(y_true, y_pred, zero_division=0), "recall": recall_score(y_true, y_pred),
               "F1": f1_score(y_true, y_pred), "MCC": matthews_corrcoef(y_true, y_pred),
               "ROC AUC": roc_auc_score(y_true, y_score), "PR AUC": average_precision_score(y_true, y_score)}
    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
        plot_confusion(confusion_matrix(y_true, y_pred), list(labels), axes[0], f"Confusion matrix – {name}")
        fpr, tpr, _ = roc_curve(y_true, y_score)
        axes[1].plot(fpr, tpr, color=RED, lw=2.4, label=f"AUC = {metrics['ROC AUC']:.3f}"); axes[1].plot([0, 1], [0, 1], color=GRAY, ls="--")
        axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate"); axes[1].set_title("ROC curve"); axes[1].legend(loc="lower right")
        p, r, _ = precision_recall_curve(y_true, y_score)
        axes[2].plot(r, p, color=DARK, lw=2.4, label=f"AP = {metrics['PR AUC']:.3f}"); axes[2].axhline(y_true.mean(), color=GRAY, ls="--")
        axes[2].set_xlabel("recall"); axes[2].set_ylabel("precision"); axes[2].set_title("Precision–Recall curve"); axes[2].legend(loc="lower left")
        plt.tight_layout(); plt.show()
    return pd.Series(metrics, name=name)

def report_multiclass(y_true, y_pred, y_proba, labels, name="model", plot=True):
    """Confusion matrix, one-vs-rest ROC curves and per-class F1 + a Series with the main metrics."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    k = len(labels)
    metrics = {"accuracy": accuracy_score(y_true, y_pred), "balanced acc.": balanced_accuracy_score(y_true, y_pred),
               "macro precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
               "macro recall": recall_score(y_true, y_pred, average="macro"), "macro F1": f1_score(y_true, y_pred, average="macro"),
               "MCC": matthews_corrcoef(y_true, y_pred), "ROC AUC (ovr)": roc_auc_score(y_true, y_proba, multi_class="ovr"),
               "log loss": log_loss(y_true, y_proba, labels=list(range(k)))}
    if plot:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), gridspec_kw={"width_ratios": [1.1, 1, 1]})
        plot_confusion(confusion_matrix(y_true, y_pred, labels=list(range(k))), list(labels), axes[0], f"Confusion matrix – {name}")
        cmap = plt.get_cmap("tab10")
        for c in range(k):
            fpr, tpr, _ = roc_curve((y_true == c).astype(int), y_proba[:, c])
            axes[1].plot(fpr, tpr, color=cmap(c), lw=1.6, label=f"{labels[c]}")
        axes[1].plot([0, 1], [0, 1], color=GRAY, ls="--"); axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate")
        axes[1].set_title(f"ROC one-vs-rest (macro AUC = {metrics['ROC AUC (ovr)']:.3f})"); axes[1].legend(fontsize=7, ncol=2 if k > 5 else 1, loc="lower right")
        f1s = f1_score(y_true, y_pred, average=None)
        axes[2].bar(range(k), f1s, color=RED, width=0.65); axes[2].set_xticks(range(k)); axes[2].set_xticklabels(labels, rotation=45 if k > 4 else 0, ha="right" if k > 4 else "center")
        axes[2].set_ylim(0, 1.05); axes[2].set_ylabel("F1"); axes[2].set_title("F1 per class")
        plt.tight_layout(); plt.show()
    return pd.Series(metrics, name=name)
def annotate_heatmap(ax, mat, fmt="{:.2f}"):
    """Write the values on an imshow heat map with a readable text color."""
    mat = np.asarray(mat, dtype=float)
    lo, hi = np.nanmin(mat), np.nanmax(mat)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            ax.text(j, i, "–" if np.isnan(v) else fmt.format(v), ha="center", va="center", color="white" if (not np.isnan(v) and v > (lo + hi) / 2) else DARK)

### **1. Dataset and Preparation**

The same problem and the same **70/30 stratified split** of the other notebooks of the series: **malignant ( 1 ) × benign ( 0 )** tumours, the 10 *mean* attributes of the Breast Cancer dataset.

In [4]:
#Load the Breast Cancer Dataset ( Positive Class = Malignant )
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer(as_frame=True)
cols = [c for c in data.data.columns if c.startswith("mean ")]            # 10 "mean" attributes
X = data.data[cols]
y = (data.target == 0).astype(int)                                        # 1 = malignant, 0 = benign
CLASS_NAMES = ["benign", "malignant"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)
print("train:", X_train.shape, "| test:", X_test.shape, "| malignant in train:", f"{y_train.mean():.1%}")
X_train.head(3)

train: (398, 10) | test: (171, 10) | malignant in train: 37.2%


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension
78,20.18,23.97,143.70,1245.0,0.129,0.345,0.375,0.160,0.291,0.081
330,16.03,15.51,105.80,793.2,0.095,0.137,0.120,0.070,0.178,0.060
378,13.66,15.15,88.27,580.6,0.083,0.075,0.042,0.025,0.179,0.059


In [5]:
#Discretise the Attributes ( Quantile Bins Fitted on the Training Set )
from sklearn.preprocessing import KBinsDiscretizer

disc = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="quantile", quantile_method="averaged_inverted_cdf").fit(X_train)
LEVELS = np.array(["low", "mid", "high"])

def to_categorical(Xnum):
    codes = disc.transform(Xnum).astype(int)
    return pd.DataFrame(LEVELS[codes], columns=[c.replace("mean ", "") for c in Xnum.columns], index=Xnum.index)

Xc_train, Xc_test = to_categorical(X_train), to_categorical(X_test)
Xc_train.head(3)

,radius,texture,perimeter,area,smoothness,compactness,concavity,concave points,symmetry,fractal dimension
78,high,high,high,high,high,high,high,high,high,high
330,high,low,high,high,mid,high,high,high,mid,mid
378,mid,low,mid,mid,low,mid,mid,mid,mid,mid


### **2. ZeroR and OneR**

OneR builds one rule per value of the attribute: `IF attribute = value THEN class`. The attribute is chosen by the **training error**, so it is a very fast and very readable model, and a good test of how informative a single attribute is.

In [ ]:
#Dictionary Used to Collect the Test Metrics of This Notebook ( the Cells Below Add Their Results to It )
results = {}

In [ ]:
#ZeroR ( Majority Class ) and OneR ( Best Single Attribute )
from sklearn.dummy import DummyClassifier

class OneR:
    def fit(self, X, y):
        y = np.asarray(y)
        self.classes_, k = np.unique(y), len(np.unique(y))
        yi = np.searchsorted(self.classes_, y)
        self.prior_ = (np.bincount(yi, minlength=k) + 1) / (len(yi) + k)
        self.errors_ = {}
        self.tables_ = {}
        for a in X.columns:
            table = {v: np.bincount(yi[(X[a] == v).to_numpy()], minlength=k) for v in sorted(X[a].unique())}
            self.tables_[a] = table
            self.errors_[a] = sum(c.sum() - c.max() for c in table.values())
        self.attribute_ = min(self.errors_, key=self.errors_.get)                           # fewest training errors
        return self

    def predict_proba(self, X):
        table, k = self.tables_[self.attribute_], len(self.classes_)
        proba = np.tile(self.prior_, (len(X), 1))
        for v, counts in table.items():
            proba[(X[self.attribute_] == v).to_numpy()] = (counts + 1) / (counts.sum() + k)       # Laplace estimate
        return proba

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]

    def n_rules(self):
        return len(self.tables_[self.attribute_])

zeror = DummyClassifier(strategy="prior").fit(Xc_train, y_train)
results["ZeroR"] = report_binary(y_test, zeror.predict(Xc_test), zeror.predict_proba(Xc_test)[:, 1], "ZeroR", CLASS_NAMES, plot=False)
oner = OneR().fit(Xc_train, y_train)
proba_oner = oner.predict_proba(Xc_test)[:, 1]
pred_oner = oner.predict(Xc_test)
results["OneR"] = report_binary(y_test, pred_oner, proba_oner, "OneR", CLASS_NAMES)
print(f"OneR chose the attribute '{oner.attribute_}' ( {oner.errors_[oner.attribute_]} training errors in {len(y_train)} cases )")
pd.DataFrame(results).T[["accuracy", "precision", "recall", "F1", "ROC AUC", "PR AUC"]].round(3)

In [ ]:
#OneR Rules and the Training Errors of Every Attribute
rows = [{"IF": f"{oner.attribute_} = {v}", "THEN": CLASS_NAMES[oner.classes_[c.argmax()]], "covers": int(c.sum()), "precision": c.max() / c.sum()} for v, c in oner.tables_[oner.attribute_].items()]
fig, ax = plt.subplots(figsize=(8, 3.6))
errs = pd.Series(oner.errors_).sort_values()
ax.barh(errs.index[::-1], errs.values[::-1], color=[RED if a == oner.attribute_ else LIGHT for a in errs.index[::-1]]); ax.set_xlabel("training errors of the one-attribute rule"); ax.set_title("OneR: one rule set per attribute")
plt.tight_layout(); plt.show()
pd.DataFrame(rows).round(3)

### **3. PRISM**

For each class `c`, PRISM repeats: start with an empty rule, **add the condition `attribute = value` with the highest precision** `p / t` ( `p` = covered cases of class `c`, `t` = covered cases ) until the rule is pure or reaches the maximum length, then remove the covered cases of class `c`.

Attention! On noisy data the original PRISM ( no limits ) tends to learn many tiny rules that memorise the training set. Two parameters limit it: `min_cover` ( a condition is accepted only if the rule still covers at least that many cases ) and `max_len`.

In [ ]:
#PRISM Implementation ( Covering Algorithm, Precision-Driven )
class PRISM:
    def __init__(self, min_cover=5, max_len=4):
        self.min_cover, self.max_len = min_cover, max_len

    def fit(self, X, y):
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        yi = np.searchsorted(self.classes_, y)
        k = len(self.classes_)
        selectors = [(a, v) for a in X.columns for v in sorted(X[a].unique())]
        masks = {s: (X[s[0]] == s[1]).to_numpy() for s in selectors}
        self.prior_ = (np.bincount(yi, minlength=k) + 1) / (len(yi) + k)
        self.rules_ = []
        for c in np.argsort(np.bincount(yi, minlength=k)):                                 # minority class first
            uncovered = (yi == c)
            while uncovered.sum() >= self.min_cover:
                covered, conds = np.ones(len(yi), bool), []
                while len(conds) < self.max_len:
                    best, best_key = None, None
                    for s in selectors:
                        if s[0] in {a for a, _ in conds}:
                            continue
                        m = covered & masks[s]
                        t = m.sum()
                        if t < self.min_cover:
                            continue
                        p = (m & uncovered).sum()
                        key = (p / t, p)                                                  # precision, then coverage
                        if p > 0 and (best_key is None or key > best_key):
                            best, best_key = s, key
                    if best is None:
                        break
                    conds.append(best); covered &= masks[best]
                    if best_key[0] == 1.0:
                        break
                if not conds:
                    break
                counts = np.bincount(yi[covered], minlength=k)
                self.rules_.append({"conds": tuple(conds), "cls": int(c), "counts": counts, "precision": counts[c] / counts.sum()})
                uncovered &= ~covered
        return self

    def _matches(self, X):
        out = []
        for r in self.rules_:
            m = np.ones(len(X), bool)
            for a, v in r["conds"]:
                m &= (X[a] == v).to_numpy()
            out.append(m)
        return np.array(out).reshape(len(self.rules_), len(X))

    def predict_proba(self, X):
        k = len(self.classes_)
        score = np.zeros((len(X), k))
        for r, m in zip(self.rules_, self._matches(X)):
            laplace = (r["counts"][r["cls"]] + 1) / (r["counts"].sum() + k)
            score[m, r["cls"]] = np.maximum(score[m, r["cls"]], laplace)                  # best rule of each class
        fired = score.sum(axis=1) > 0
        proba = np.tile(self.prior_, (len(X), 1))
        proba[fired] = score[fired] / score[fired].sum(axis=1, keepdims=True)             # conflicts: compare the best rules of both classes
        return proba

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]

    def n_rules(self):
        return len(self.rules_)

    def rules_table(self):
        return pd.DataFrame([{"IF": " AND ".join(f"{a}={v}" for a, v in r["conds"]), "THEN": CLASS_NAMES[self.classes_[r["cls"]]], "covers": int(r["counts"].sum()),
                              "precision": r["precision"], "length": len(r["conds"])} for r in self.rules_])

prism = PRISM(min_cover=5, max_len=4).fit(Xc_train, y_train)
proba_prism = prism.predict_proba(Xc_test)[:, 1]
pred_prism = prism.predict(Xc_test)
print(f"PRISM learned {prism.n_rules()} rules ( min_cover = 5, max_len = 4 )")
results["PRISM"] = report_binary(y_test, pred_prism, proba_prism, "PRISM", CLASS_NAMES)
prism.rules_table().head(8).round(3)

In [ ]:
#Rule Quality of PRISM: Coverage and Precision on the Training Set, and How Often the Rules Conflict on the Test Set
rt = prism.rules_table()
matches = prism._matches(Xc_test)
classes_per_case = [set(prism.rules_[i]["cls"] for i in np.flatnonzero(matches[:, j])) for j in range(len(Xc_test))]
fired = np.array([len(s) > 0 for s in classes_per_case]); conflict = np.array([len(s) > 1 for s in classes_per_case])
print(f"rules: {len(rt)} | mean length: {rt['length'].mean():.2f} | rules with training precision 1.0: {(rt['precision'] == 1).sum()} | test cases covered by at least one rule: {fired.mean():.1%} | cases with rules of both classes ( conflict ): {conflict.mean():.1%}")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
colors = [RED if t == "malignant" else DARK for t in rt["THEN"]]
axes[0].scatter(rt["covers"], rt["precision"], c=colors, s=26); axes[0].set_xlabel("training cases covered"); axes[0].set_ylabel("precision"); axes[0].set_title("Rules ( red: malignant, dark: benign )")
axes[1].hist(rt["length"], bins=np.arange(0.5, 5.5), color=DARK, rwidth=0.8); axes[1].set_xlabel("conditions per rule"); axes[1].set_ylabel("rules"); axes[1].set_title("Rule length")
plt.tight_layout(); plt.show()

In [ ]:
#Sensitivity of PRISM to min_cover and max_len ( Test Accuracy and Number of Rules; Not a Tuning Procedure )
rows = [{"min_cover": mc, "max_len": ml, "rules": (m := PRISM(min_cover=mc, max_len=ml).fit(Xc_train, y_train)).n_rules(), "test accuracy": accuracy_score(y_test, m.predict(Xc_test))}
        for mc in [1, 3, 5, 10, 20] for ml in [1, 2, 3, 4]]
sens = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, col, fmt in zip(axes, ["rules", "test accuracy"], ["{:.0f}", "{:.2f}"]):
    piv = sens.pivot(index="min_cover", columns="max_len", values=col)
    ax.imshow(piv, cmap="Reds", aspect="auto"); ax.grid(False)
    ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns); ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index)
    ax.set_xlabel("max_len"); ax.set_ylabel("min_cover"); ax.set_title(col); annotate_heatmap(ax, piv.to_numpy(), fmt)
plt.tight_layout(); plt.show()

### **4. Quality Metrics**

In [ ]:
#Repeated Stratified CV on the Training Set ( 5 Folds x 3 ): the Discretiser and the Model Are Refitted Inside Every Fold
from sklearn.model_selection import RepeatedStratifiedKFold
cv_rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)

def cv_discretised(make_model):
    accs, aucs, sizes = [], [], []
    for tr, te in cv_rep.split(X_train, y_train):
        d = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="quantile", quantile_method="averaged_inverted_cdf").fit(X_train.iloc[tr])
        codes = lambda part: pd.DataFrame(LEVELS[d.transform(part).astype(int)], columns=Xc_train.columns, index=part.index)
        model = make_model().fit(codes(X_train.iloc[tr]), y_train.iloc[tr])
        p = model.predict_proba(codes(X_train.iloc[te]))[:, 1]
        accs.append(((p > 0.5).astype(int) == y_train.iloc[te].to_numpy()).mean()); aucs.append(roc_auc_score(y_train.iloc[te], p)); sizes.append(model.n_rules())
    return np.array(accs), np.array(aucs), np.array(sizes)

In [ ]:
#CV of the Three Learners ( ZeroR, OneR, PRISM ) and Which Attribute OneR Chooses in Each Fold
class ZeroR:
    def fit(self, X, y):
        self.p_ = (np.bincount(np.asarray(y), minlength=2) + 1) / (len(y) + 2)
        return self
    def predict_proba(self, X):
        return np.tile(self.p_, (len(X), 1))
    def n_rules(self):
        return 1

cv_stats = {}
for label, factory in [("ZeroR ( majority )", ZeroR), ("OneR", OneR), ("PRISM", lambda: PRISM(min_cover=5, max_len=4))]:
    a, u, s = cv_discretised(factory)
    cv_stats[label] = {"accuracy ( mean )": a.mean(), "accuracy ( std )": a.std(), "ROC AUC ( mean )": u.mean(), "ROC AUC ( std )": u.std(), "rules ( mean )": s.mean()}
chosen = []
for tr, te in cv_rep.split(X_train, y_train):
    d = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="quantile", quantile_method="averaged_inverted_cdf").fit(X_train.iloc[tr])
    chosen.append(OneR().fit(pd.DataFrame(LEVELS[d.transform(X_train.iloc[tr]).astype(int)], columns=Xc_train.columns), y_train.iloc[tr]).attribute_)
print("attribute chosen by OneR in the 15 folds:", pd.Series(chosen).value_counts().to_dict())
cv_table = pd.DataFrame(cv_stats).T
cv_table.round(3)

In [ ]:
#Reference Models on the Same Split ( Is the Technique Better than a Trivial or a Standard Model? )
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

references = {"Majority class": DummyClassifier(strategy="prior"),
              "Decision tree ( depth 3 )": DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
              "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))}
reference_rows = {}
for ref_name, model in references.items():
    model.fit(X_train, y_train)
    reference_rows[ref_name] = report_binary(y_test, model.predict(X_test), model.predict_proba(X_test)[:, 1], ref_name, CLASS_NAMES, plot=False)
metric_cols = ["accuracy", "precision", "recall", "F1", "ROC AUC", "PR AUC"]
pd.concat([pd.DataFrame(results).T[metric_cols], pd.DataFrame(reference_rows).T[metric_cols]]).round(3)

In [ ]:
#Same Repeated CV for the Reference Models ( Numeric Attributes, Refitted in Every Fold )
from sklearn.model_selection import cross_validate
cv_rows = {}
for ref_name, model in references.items():
    res = cross_validate(model, X_train, y_train, cv=cv_rep, scoring=["accuracy", "roc_auc"])
    cv_rows[ref_name] = {"accuracy ( mean )": res["test_accuracy"].mean(), "accuracy ( std )": res["test_accuracy"].std(), "ROC AUC ( mean )": res["test_roc_auc"].mean(), "ROC AUC ( std )": res["test_roc_auc"].std()}
cv_table = pd.concat([cv_table[["accuracy ( mean )", "accuracy ( std )", "ROC AUC ( mean )", "ROC AUC ( std )"]], pd.DataFrame(cv_rows).T]).round(3)
cv_table

In [ ]:
#Uncertainty of the Test Metrics ( Bootstrap of the Test Cases, 1,000 Resamples )
def bootstrap_ci(y_true, pred, score, n_boot=1000, seed=RANDOM_STATE):
    rng_b = np.random.default_rng(seed)
    y_true, pred, score = np.asarray(y_true), np.asarray(pred), np.asarray(score)
    acc, auc = [], []
    for _ in range(n_boot):
        i = rng_b.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[i])) < 2:
            continue
        acc.append((pred[i] == y_true[i]).mean())
        auc.append(roc_auc_score(y_true[i], score[i]))
    return pd.DataFrame({"accuracy": np.percentile(acc, [2.5, 50, 97.5]), "ROC AUC": np.percentile(auc, [2.5, 50, 97.5])}, index=["2.5%", "median", "97.5%"]).T

print("the test set has", len(y_test), "cases: differences smaller than these intervals are not reliable")
bootstrap_ci(y_test, pred_prism, proba_prism).round(3)

In [ ]:
#Calibration of the Predicted Probabilities ( Brier Score and Reliability Curve )
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve

proba_prism_ARRAY = np.asarray(proba_prism, dtype=float)
frac_pos, mean_pred = calibration_curve(y_test, proba_prism_ARRAY, n_bins=5, strategy="quantile")                       # bins with the same number of cases
print(f"Brier score: {brier_score_loss(y_test, proba_prism_ARRAY):.4f} | Brier score of a model that always predicts the training prevalence: {brier_score_loss(y_test, np.full(len(y_test), y_train.mean())):.4f} ( lower is better )")
fig, ax = plt.subplots(figsize=(4.6, 4.2))
ax.plot([0, 1], [0, 1], color=GRAY, ls="--", label="perfect calibration"); ax.plot(mean_pred, frac_pos, color=RED, marker="o", label="PRISM")
ax.set_xlabel("mean predicted probability ( 5 bins of equal size )"); ax.set_ylabel("observed share of malignant"); ax.set_title("Reliability curve"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### **5. Takeaways**

* **ZeroR** is the floor; **OneR** shows how far one attribute goes: the distance between OneR and PRISM is the value of **combining conditions**
* **OneR is fairly stable** ( the same attribute in most of the folds, see the count above ) and very readable, but it cannot use interactions
* **More rules do not mean more accuracy:** in the sensitivity grid the loosest setting learns 66 rules and is not better on the test set than settings with a dozen rules ( limited by `min_cover` and `max_len` ); the extra rules mostly memorise the training set. The grid is a sensitivity analysis, not a tuning procedure
* PRISM rules are **unordered** and can conflict; the conflict rate is reported and resolved by comparing the best rule of each class
* The discretiser was **refitted inside every fold** in the cross-validation; with 171 test cases, differences smaller than the bootstrap intervals are not reliable